# Containers and Nodes

Before any equipment, a model needs its base quantities, the containers equipment goes in, and the nodes
it connects to. This page covers the builders for those.

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs. `new_feeder_model` makes an empty model to build into.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    BasePowerBuilder,
    BaseVoltageBuilder,
    BayBuilder,
    BusbarSectionBuilder,
    ConnectivityNodeBuilder,
    EnergyConsumerBuilder,
    FeederBuilder,
    JuntionBuilder,
    SubstationBuilder,
    TopologicalNodeBuilder,
    VoltageLevelBuilder,
)

network = new_feeder_model(
    authority="https://example.com/models",
    model="containers_and_nodes",
    Feeder="fdr_demo",
)
container = network.container

## Base quantities

`BaseVoltageBuilder`, `BasePowerBuilder` and `BaseFrequencyBuilder` take their value straight in
`create()`. `new_feeder_model` has already made the `BaseFrequency` `base_freq`.

In [2]:
base_voltages = BaseVoltageBuilder(network, container)
base_voltages.create(mRID="bv_115kV", name="115 kV", nominalVoltage=Qty(115, "kV"))
base_voltages.create(
    mRID="bv_12_47kV", name="12.47 kV", nominalVoltage=Qty(12.47, "kV")
)
base_power = BasePowerBuilder(network, container).create(
    mRID="base_power", basePower=Qty(100, "MVA")
)

nominal_voltage = network.find_by_mRID("bv_12_47kV").nominalVoltage
print(f"{nominal_voltage.to('kV'):g} kV, {base_power.basePower.to('MVA'):g} MVA")

12.47 kV, 100 MVA


## Feeders and substations

`new_feeder_model` makes one Feeder, the graph's container. More feeders are made with `FeederBuilder`,
and `add_connectivity` links a feeder to the substation that normally energizes it. Equipment and nodes go
in a feeder other than the builder's container through their `EquipmentContainer` and
`ConnectivityNodeContainer` arguments.

In [3]:
substation_builder = SubstationBuilder(network, container)
substation = substation_builder.create(mRID="sub_main", name="main")

feeder_builder = FeederBuilder(network, container)
feeder_builder.add_connectivity(container, NormalEnergizingSubstation="sub_main")
feeder_b = feeder_builder.create(mRID="fdr_b", name="feeder b")
feeder_builder.add_connectivity(feeder_b, NormalEnergizingSubstation="sub_main")

print(sorted(feeder.mRID for feeder in substation.NormalEnergizedFeeder))

['fdr_b', 'fdr_demo']


## Nodes

A `ConnectivityNode` is a point that Terminals connect to. Building one with `ConnectivityNodeBuilder`
gives it a name, a `BaseVoltage` and a container. Equipment that refers to a node mRID that does not exist
yet creates a bare node, with neither a name nor a `BaseVoltage`, in the equipment's container.

In [4]:
node_builder = ConnectivityNodeBuilder(network, container)
node = node_builder.create(mRID="b1", name="b1")
node_builder.add_connectivity(
    node, BaseVoltage="bv_12_47kV", ConnectivityNodeContainer="fdr_b"
)

load_builder = EnergyConsumerBuilder(network, container)
load = load_builder.create(mRID="ld_b1")
load_builder.add_connectivity(
    load,
    node="b1",
    phases="ABC",
    EquipmentContainer="fdr_b",
    BaseVoltage="bv_12_47kV",
)
load_builder.add_connectivity(
    load_builder.create(mRID="ld_new"), node="new_node", BaseVoltage="bv_12_47kV"
)

for mRID in ("b1", "new_node"):
    node = network.find_by_mRID(mRID)
    node_container = node.ConnectivityNodeContainer.mRID
    base_voltage = getattr(node.BaseVoltage, "name", None)
    print(
        f"{mRID:9} name={node.name!s:5}",
        f"container={node_container:9}",
        f"base voltage={base_voltage}",
    )
print("ld_b1 is in", load.EquipmentContainer.mRID)

b1        name=b1    container=fdr_b     base voltage=12.47 kV
new_node  name=None  container=fdr_demo  base voltage=None
ld_b1 is in fdr_b


## Voltage levels and bays

Inside a substation, equipment goes in a `VoltageLevel`, one per nominal voltage, and optionally a `Bay`
within it. The [substation builder](../02_substation_builder/2_1_substation_builder.ipynb) builds whole
node-breaker layouts from these. Here they are by hand:

In [5]:
voltage_level_builder = VoltageLevelBuilder(network, container)
voltage_level = voltage_level_builder.create(mRID="vl_115", name="115 kV")
voltage_level_builder.add_connectivity(
    voltage_level, Substation="sub_main", BaseVoltage="bv_115kV"
)

bay_builder = BayBuilder(network, container)
bay = bay_builder.create(mRID="bay_1", name="line bay 1")
bay_builder.add_connectivity(
    bay,
    VoltageLevel="vl_115",
    breakerConfiguration="singleBreaker",
    busBarConfiguration="singleBus",
)

print(voltage_level.Substation.mRID, [b.mRID for b in voltage_level.Bays])

sub_main ['bay_1']


## Busbars and junctions

A `BusbarSection` or a `Junction` has one Terminal, on the node it marks. Pass the `VoltageLevel` as the
`EquipmentContainer` to put it in the substation.

In [6]:
busbar_builder = BusbarSectionBuilder(network, container)
busbar = busbar_builder.create(mRID="vl_115_bus_1_bbs", name="bus 1")
busbar, terminal = busbar_builder.add_connectivity(
    busbar, node="vl_115_bus_1", EquipmentContainer="vl_115", BaseVoltage="bv_115kV"
)
busbar_builder.add_short_circuit(busbar, ipMax=Qty(40, "kA"))

junction_builder = JuntionBuilder(network, container)
junction = junction_builder.create(mRID="j_tap_1")
junction_builder.add_connectivity(junction, node="tap_1", BaseVoltage="bv_12_47kV")

print(
    terminal.mRID,
    "on",
    terminal.ConnectivityNode.mRID,
    "in",
    busbar.EquipmentContainer.mRID,
)
print(f"ipMax = {busbar.ipMax.to('kA'):g} kA")

vl_115_bus_1_bbs_T1 on vl_115_bus_1 in vl_115
ipMax = 40 kA


## Topological nodes

A bus-branch model, such as one translated from PSS/E, has `TopologicalNode`s instead of, or as well as,
ConnectivityNodes. `TopologicalNodeBuilder` adds the bus number and name a planning model keys its buses
by.

In [7]:
topological_node_builder = TopologicalNodeBuilder(network, container)
bus = topological_node_builder.create(mRID="tn_101", name="BUS 101")
topological_node_builder.add_bus_number(bus, busName="BUS 101", busNumber=101)
print(bus.mRID, bus.busNumber)

tn_101 101
